# Week 8 — Bayesian Networks and Autoregressive Language Models

**CS F407 Artificial Intelligence · Lab 8**

This notebook builds a first-order and a second-order autoregressive language model as Bayesian networks, estimates their conditional probability tables (CPTs) by counting, tests them, and uses them to predict and generate text. Answers to Questions 1–14 are written inline next to the part of the lab they belong to, and are also collected in [`answers-to-questions/answers.md`](../answers-to-questions/answers.md).

Everything uses plain Python (`collections`, `random`, `fractions`): no ML libraries and no pretrained models, as the lab asks.

| Part | Contents |
|---|---|
| I–II | Chain rule, first-order Markov BN (Q1, Q2) |
| III–IV | Dataset, transition counts, CPT (Q3) |
| V–VII | LLM-assisted implementation, code inspection, normalisation tests (Q4–Q8) |
| VIII–X | Next-word prediction, generation, greedy vs sampling (Q9, Q10) |
| XI–XIII | Second-order BN, comparison of the two models (Q11, Q12) |
| XIV–XV | Connection to modern LMs, reflection on using the LLM (Q13, Q14) |

In [1]:
import random
from collections import Counter, defaultdict
from fractions import Fraction
from pathlib import Path

START, END = "<START>", "<END>"
SEED = 407            # fixed seed so the sampled outputs below are reproducible
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

## Part I — From Probability to Language

For the sentence `the cat sat on the mat`, the chain rule gives the exact factorisation

$$P(X_1,\dots,X_6) = P(X_1)\,P(X_2\mid X_1)\,P(X_3\mid X_1,X_2)\,P(X_4\mid X_1,X_2,X_3)\,P(X_5\mid X_1,\dots,X_4)\,P(X_6\mid X_1,\dots,X_5).$$

No independence assumption has been made yet; this identity holds for any joint distribution.

### Question 1 — Why is this decomposition useful for generating text?

- **It turns one impossible distribution into many small ones.** A joint distribution over whole sentences has $|V|^T$ outcomes, far too many to write down or sample from directly. Each chain-rule factor is only a distribution over the vocabulary $V$: "which word comes next, given what has been written so far".
- **Generation becomes sequential sampling.** Sample $x_1 \sim P(X_1)$, then $x_2 \sim P(X_2\mid x_1)$, then $x_3 \sim P(X_3 \mid x_1, x_2)$, and so on. Because the product of the factors *is* the joint, this ancestral sampling draws whole sentences from exactly $P(X_1,\dots,X_T)$. Text is produced left to right, the way it is written.
- **Variable length comes for free.** Adding an `<END>` token lets the model decide at every step whether to stop, so it never needs to know $T$ in advance.
- **Each factor can be learned from data** as a next-word prediction problem, and the same factors also **score** any given sentence (its probability is the product of the factors), so one model supports both generation and evaluation.

## Part II — A Bayesian Network for Text

If each word depends only on the immediately preceding word, the BN is a chain

$$X_1 \rightarrow X_2 \rightarrow X_3 \rightarrow X_4, \qquad P(X_1,X_2,X_3,X_4) = P(X_1)\,P(X_2\mid X_1)\,P(X_3\mid X_2)\,P(X_4\mid X_3).$$

### Question 2 — What independence assumption is being made?

The **first-order Markov assumption**: given the previous word, the current word is conditionally independent of all earlier words.

$$X_t \;\perp\; \{X_1,\dots,X_{t-2}\} \;\mid\; X_{t-1}
\qquad\Longleftrightarrow\qquad
P(X_t \mid X_1,\dots,X_{t-1}) = P(X_t \mid X_{t-1}).$$

For example, $P(\texttt{mat}\mid \texttt{the cat sat on the}) = P(\texttt{mat}\mid\texttt{the})$.

Two further points:

- The independence is **conditional**, not marginal. $X_t$ and $X_{t-2}$ are still dependent through the chain $X_{t-2}\to X_{t-1}\to X_t$; observing $X_{t-1}$ blocks that path (d-separation).
- We also use **one CPT for every position** $t$ (time-homogeneity / parameter tying): $P(X_t = w \mid X_{t-1} = v)$ is the same whether we are at word 2 or word 5. That is what lets six short sentences estimate the whole model.

## Part III — Build a Small Language Dataset

We use the dataset from the lab sheet, convert it to lower case, split it on whitespace (one word = one token), and add `<START>` and `<END>` markers.

In [2]:
raw_sentences = [
    "the cat sat on the mat",
    "the cat sat on the rug",
    "the dog sat on the mat",
    "the dog ran to the park",
    "the cat ran to the park",
    "the dog sat on the rug",
]

def tokenise(sentence):
    return sentence.lower().split()

def add_markers(tokens, order=1):
    # An order-k model needs k <START> tokens so the first word has a full context.
    return [START] * order + tokens + [END]

tokenised = [tokenise(s) for s in raw_sentences]
training_data = [add_markers(t) for t in tokenised]

for s in training_data:
    print(" ".join(s))

vocab = sorted({w for t in tokenised for w in t})
print(f"\n{len(vocab)} word types:", vocab)

<START> the cat sat on the mat <END>
<START> the cat sat on the rug <END>
<START> the dog sat on the mat <END>
<START> the dog ran to the park <END>
<START> the cat ran to the park <END>
<START> the dog sat on the rug <END>

10 word types: ['cat', 'dog', 'mat', 'on', 'park', 'ran', 'rug', 'sat', 'the', 'to']


## Part IV — Constructing the Conditional Probability Table

The maximum-likelihood estimate of each CPT entry is a normalised count:

$$P(w_j \mid w_i) = \frac{C(w_i, w_j)}{\sum_k C(w_i, w_k)}.$$

Counting by hand first, so the implementation can be checked against it later. `the` appears 12 times as a predecessor (twice per sentence): `the cat` ×3, `the dog` ×3, `the mat` ×2, `the rug` ×2, `the park` ×2. So $P(\texttt{cat}\mid\texttt{the}) = 3/12 = 0.25$, not the $3/5$ of the lab sheet's illustrative example, because `the` also precedes the object noun.

## Part V — Ask an LLM to Implement the Model

I gave the LLM a behavioural specification rather than "write a language model in Python". The prompt I used:

> Write a simple Python implementation of a first-order autoregressive language model. The model should:
> 1. take a list of tokenised sentences (already padded with `<START>` and `<END>`) as training data;
> 2. count transitions between consecutive tokens and store them as `counts[prev][next]`;
> 3. construct the conditional distribution $P(X_t \mid X_{t-1})$ by normalising each row of counts;
> 4. display the probabilities for a specified previous token;
> 5. predict the most probable next token, **reporting ties instead of hiding them**;
> 6. generate a sentence by repeatedly sampling the next token, with a `mode` switch for greedy (argmax) vs sampling;
> 7. stop when the `<END>` token is generated, **or after `max_tokens` steps**, and say which of the two happened;
> 8. return an empty distribution (not raise) for a context that was never observed.
>
> Do not use a machine-learning library or a pretrained language model. Use ordinary Python data structures and `random` with an explicit `random.Random` instance so runs are reproducible.

Items 5, 7 (the `max_tokens` part) and 8 were **not** in my first prompt. I added them after inspecting and testing the first version (see Part X and the reflection at the end). The code below is the version I accepted.

In [3]:
class FirstOrderLM:
    # First-order autoregressive LM, i.e. the Bayesian network X_{t-1} -> X_t.

    order = 1

    def __init__(self):
        self.counts = defaultdict(Counter)   # counts[prev][next] = C(prev, next)
        self.probs = {}                      # probs[prev][next] = P(next | prev)

    def context(self, history):
        return history[-1]

    def fit(self, sentences):
        for tokens in sentences:
            for prev, nxt in zip(tokens[:-1], tokens[1:]):
                self.counts[prev][nxt] += 1
        # Normalise each row of counts into a conditional distribution.
        for ctx, row in self.counts.items():
            total = sum(row.values())
            self.probs[ctx] = {w: c / total for w, c in row.items()}
        return self

    def distribution(self, ctx):
        # P(. | ctx); an empty dict means the context was never observed.
        return self.probs.get(ctx, {})

    def show(self, ctx):
        row = self.counts.get(ctx)
        if not row:
            print(f"P(. | {ctx}): context never observed")
            return
        total = sum(row.values())
        print(f"P(. | {ctx})   [{total} observations]")
        for w, c in sorted(row.items(), key=lambda kv: (-kv[1], kv[0])):
            print(f"   {w:<8} {c:>2}/{total:<2} = {c / total:.4f}")

    def argmax(self, ctx):
        # All next tokens that share the highest probability (ties are common on tiny data).
        dist = self.distribution(ctx)
        if not dist:
            return []
        best = max(dist.values())
        return sorted(w for w, p in dist.items() if p == best)

    def predict(self, ctx):
        # Most probable next token; ties are broken alphabetically so the result is deterministic.
        best = self.argmax(ctx)
        return best[0] if best else None

    def sample_next(self, ctx, rng):
        dist = self.distribution(ctx)
        if not dist:
            return None
        words = list(dist)
        return rng.choices(words, weights=[dist[w] for w in words], k=1)[0]

    def generate(self, mode="sample", max_tokens=20, rng=None):
        # Returns (words, finished); finished is False if max_tokens was hit before <END>.
        rng = rng or random.Random()
        history = [START] * self.order
        words = []
        for _ in range(max_tokens):
            ctx = self.context(history)
            nxt = self.predict(ctx) if mode == "greedy" else self.sample_next(ctx, rng)
            if nxt is None or nxt == END:
                return words, nxt == END
            words.append(nxt)
            history.append(nxt)
        return words, False


lm1 = FirstOrderLM().fit(training_data)

### Question 3 — The CPT $P(\text{next word}\mid\text{current word})$

First the requested rows (`the`, `cat`, `dog`, `sat`, `ran`), then the full table, where `·` marks a zero-probability transition.

In [4]:
for w in ["the", "cat", "dog", "sat", "ran"]:
    lm1.show(w)
    print()

P(. | the)   [12 observations]
   cat       3/12 = 0.2500
   dog       3/12 = 0.2500
   mat       2/12 = 0.1667
   park      2/12 = 0.1667
   rug       2/12 = 0.1667

P(. | cat)   [3 observations]
   sat       2/3  = 0.6667
   ran       1/3  = 0.3333

P(. | dog)   [3 observations]
   sat       2/3  = 0.6667
   ran       1/3  = 0.3333

P(. | sat)   [4 observations]
   on        4/4  = 1.0000

P(. | ran)   [2 observations]
   to        2/2  = 1.0000



In [5]:
def print_cpt(model, contexts, columns, fmt=lambda ctx: str(ctx)):
    width = max(len(fmt(c)) for c in contexts) + 2
    print(" " * width + "".join(f"{c:>7}" for c in columns))
    for ctx in contexts:
        row = model.counts.get(ctx, {})
        total = sum(row.values())
        cells = "".join(f"{str(Fraction(row[c], total)) if row.get(c) else '·':>7}" for c in columns)
        print(f"{fmt(ctx):<{width}}{cells}")

contexts_1 = [START] + vocab
next_tokens = vocab + [END]
print_cpt(lm1, contexts_1, next_tokens)

             cat    dog    mat     on   park    ran    rug    sat    the     to  <END>
<START>        ·      ·      ·      ·      ·      ·      ·      ·      1      ·      ·
cat            ·      ·      ·      ·      ·    1/3      ·    2/3      ·      ·      ·
dog            ·      ·      ·      ·      ·    1/3      ·    2/3      ·      ·      ·
mat            ·      ·      ·      ·      ·      ·      ·      ·      ·      ·      1
on             ·      ·      ·      ·      ·      ·      ·      ·      1      ·      ·
park           ·      ·      ·      ·      ·      ·      ·      ·      ·      ·      1
ran            ·      ·      ·      ·      ·      ·      ·      ·      ·      1      ·
rug            ·      ·      ·      ·      ·      ·      ·      ·      ·      ·      1
sat            ·      ·      ·      1      ·      ·      ·      ·      ·      ·      ·
the          1/4    1/4    1/6      ·    1/6      ·    1/6      ·      ·      ·      ·
to             ·      ·      ·      ·      

In [6]:
# Zero-probability transitions: every (current, next) pair with C(current, next) = 0.
print("Zero-probability transitions from the requested words:")
for w in ["the", "cat", "dog", "sat", "ran"]:
    zeros = [v for v in next_tokens if v not in lm1.probs[w]]
    print(f"  {w:<4} -> {len(zeros):>2} of {len(next_tokens)} zero: {zeros}")

n_cells = len(contexts_1) * len(next_tokens)
n_zero = sum(1 for c in contexts_1 for v in next_tokens if v not in lm1.distribution(c))
print(f"\nWhole table: {n_zero} of {n_cells} entries ({n_zero / n_cells:.0%}) are zero; "
      f"only {n_cells - n_zero} transitions were ever observed.")

Zero-probability transitions from the requested words:
  the  ->  6 of 11 zero: ['on', 'ran', 'sat', 'the', 'to', '<END>']
  cat  ->  9 of 11 zero: ['cat', 'dog', 'mat', 'on', 'park', 'rug', 'the', 'to', '<END>']
  dog  ->  9 of 11 zero: ['cat', 'dog', 'mat', 'on', 'park', 'rug', 'the', 'to', '<END>']
  sat  -> 10 of 11 zero: ['cat', 'dog', 'mat', 'park', 'ran', 'rug', 'sat', 'the', 'to', '<END>']
  ran  -> 10 of 11 zero: ['cat', 'dog', 'mat', 'on', 'park', 'ran', 'rug', 'sat', 'the', '<END>']

Whole table: 104 of 121 entries (86%) are zero; only 17 transitions were ever observed.


**Reading the CPT**

| current word | next-word distribution | zero-probability transitions (examples) |
|---|---|---|
| `the` | cat 1/4, dog 1/4, mat 1/6, rug 1/6, park 1/6 | `the the`, `the sat`, `the on`, `the <END>` |
| `cat` | sat 2/3, ran 1/3 | `cat on`, `cat to`, `cat <END>`, `cat the` |
| `dog` | sat 2/3, ran 1/3 | same as `cat`: the two words have identical rows |
| `sat` | on 1 | everything except `on`, e.g. `sat to`, `sat down` |
| `ran` | to 1 | everything except `to`, e.g. `ran on` |

Zero-probability transitions:

- Most of the table is zero: 104 of 121 entries (86%). Only 17 distinct transitions were observed in six sentences.
- Some zeros are **genuinely impossible** in English (`the the`, `the <END>`, `cat the`), so the model is right to reject them. The converse also happens: some *non-zero* paths are not English. `<START> → the → mat → <END>` gives the "sentence" `the mat`, because `mat → <END>` was learned from the end of a longer sentence.
- Others are **perfectly grammatical but unseen**: `cat on` ("the cat on the mat"), `sat to`, `ran on`, `dog <END>` ("the dog"). The maximum-likelihood estimate cannot tell "impossible" from "not seen yet". Any sentence containing one of these gets joint probability exactly 0.
- `sat → on` and `ran → to` are deterministic (probability 1), so the model can never produce "ran on the mat".
- `<END>` never appears as a context. It is absorbing: nothing follows it, which is what stops generation.

## Part VI — Inspect the LLM-Generated Code

### Question 4 — Where are the transition counts stored?
In `self.counts`, a `defaultdict(Counter)` created in `FirstOrderLM.__init__` and filled in `fit`. `self.counts[prev][nxt]` is $C(\text{prev}, \text{nxt})$. It is incremented once for each consecutive pair produced by `zip(tokens[:-1], tokens[1:])`, which covers every transition including `<START> → the` and `mat → <END>`.

### Question 5 — Where is $P(X_t \mid X_{t-1})$ computed?
At the end of `fit`, in the normalisation loop: for each context the row total $\sum_k C(\text{ctx}, w_k)$ is computed and every count is divided by it, giving `self.probs[ctx][w]` $= C(\text{ctx}, w)/\sum_k C(\text{ctx}, w_k)$. `distribution(ctx)` just looks the row up. The probabilities are computed once, not recomputed on every call.

### Question 6 — How does the program choose the next word?
It supports both, selected by `mode` in `generate`:

1. **`mode="greedy"`** calls `predict`, which always takes $\arg\max_w P(w\mid\text{ctx})$ (ties broken alphabetically).
2. **`mode="sample"`** (the default) calls `sample_next`, which draws $w \sim P(\cdot\mid\text{ctx})$ with `rng.choices(words, weights=probs)`.

The difference: greedy is **deterministic**. The same context always gives the same word, so it only ever produces one sentence and ignores every lower-probability alternative. Sampling is **stochastic**. A word with probability 1/3 is chosen about a third of the time, so repeated runs give different sentences whose frequencies match the model's distribution. Only sampling draws from the joint $P(X_1,\dots,X_T)$ that the BN defines. Greedy returns a single, locally-best path, which need not be the most probable sentence (Part X shows it may never finish).

### Question 7 — What happens for a word with no observed transitions?

In [7]:
for ctx in ["bird", END]:
    print(f"context {ctx!r}: distribution={lm1.distribution(ctx)}, "
          f"predict={lm1.predict(ctx)}, sample={lm1.sample_next(ctx, random.Random(SEED))}")
lm1.show("bird")

context 'bird': distribution={}, predict=None, sample=None
context '<END>': distribution={}, predict=None, sample=None
P(. | bird): context never observed


There are two cases:

- **Unseen context** (an out-of-vocabulary word like `bird`, or `<END>`): there is no row in the CPT, so $P(\cdot\mid\text{bird})$ is undefined (a division 0/0). The implementation returns an empty distribution, `predict` and `sample_next` return `None`, and `generate` stops and reports `finished=False`. A naive `probs[word]` would raise a `KeyError` instead. For `<END>` this behaviour is intended: generation is supposed to stop there.
- **Seen context, unseen transition** (e.g. `cat → on`): the entry is 0, so that word can never be generated after `cat`, and any test sentence that contains the pair gets probability 0.

The standard remedies are **smoothing** (add-$k$/Laplace: $\frac{C(v,w)+k}{C(v)+k|V|}$), **back-off / interpolation** to a lower-order (unigram) distribution, and an **`<UNK>` token** for words outside the vocabulary. They are not used here, so the CPT stays the pure maximum-likelihood estimate from the lab sheet.

## Part VII — Test the Probability Model

For every context $w$ the CPT row must be a probability distribution: $\sum_v P(v\mid w) = 1$ and $0 \le P(v \mid w) \le 1$. The test below checks this and fails loudly instead of just printing.

In [8]:
def check_normalisation(probs, tol=1e-9, fmt=str):
    ok = True
    for ctx, dist in probs.items():
        total = sum(dist.values())
        valid = abs(total - 1.0) < tol and all(0.0 <= p <= 1.0 for p in dist.values())
        ok &= valid
        print(f"  {fmt(ctx):<22} sum = {total:.12f}  {'OK' if valid else 'FAIL'}")
    return ok

assert check_normalisation(lm1.probs), "first-order CPT is not normalised"
print("\nAll first-order rows sum to 1.")

  <START>                sum = 1.000000000000  OK
  the                    sum = 1.000000000000  OK
  cat                    sum = 1.000000000000  OK
  sat                    sum = 1.000000000000  OK
  on                     sum = 1.000000000000  OK
  mat                    sum = 1.000000000000  OK
  rug                    sum = 1.000000000000  OK
  dog                    sum = 1.000000000000  OK
  ran                    sum = 1.000000000000  OK
  to                     sum = 1.000000000000  OK
  park                   sum = 1.000000000000  OK

All first-order rows sum to 1.


In [9]:
# Consistency with the hand calculation from Part IV.
assert lm1.counts["the"] == Counter(cat=3, dog=3, mat=2, rug=2, park=2)
assert lm1.probs["the"]["cat"] == 3 / 12 and lm1.probs["cat"]["sat"] == 2 / 3
assert lm1.probs[START] == {"the": 1.0} and lm1.probs["sat"] == {"on": 1.0}
# Total transitions = sum of (sentence length + 1) over the corpus.
assert sum(map(sum, (r.values() for r in lm1.counts.values()))) == sum(len(t) + 1 for t in tokenised)
print("Counts and probabilities match the hand-computed values.")

Counts and probabilities match the hand-computed values.


### Question 8 — A row sums to 0.87. What does that tell you?

It means the implementation is **wrong**, not just imprecise. Floating-point error is around $10^{-16}$, not $0.13$. The row is not a probability distribution: 13% of the probability mass for that context has gone missing, so the counts and the normaliser disagree. Typical causes:

- the **denominator counts more than the numerators**, e.g. dividing by the unigram count $C(w)$ including positions where $w$ has no successor (sentence-final words when `<END>` is not added), or smoothing the denominator ($+k|V|$) without adding $k$ to the numerators;
- **transitions dropped while counting**, e.g. an off-by-one loop `range(len(tokens) - 2)` that skips the last pair (`mat → <END>`), or successor words that were filtered out of the row but still counted in the total;
- **truncation/rounding** of stored probabilities.

This bug is easy to miss because `random.choices` silently renormalises its weights, so sampling still "works". But the model's probabilities, sentence likelihoods, and any comparison between models would all be wrong. A buggy variant shows the test catching it:

In [10]:
# Deliberately buggy variant: smoothing applied to the denominator only.
V = len(next_tokens)
k = 0.1
buggy_probs = {ctx: {w: c / (sum(row.values()) + k * V) for w, c in row.items()}
               for ctx, row in lm1.counts.items()}
print("Buggy model:")
print("\nTest passed?", check_normalisation(buggy_probs))

Buggy model:
  <START>                sum = 0.845070422535  FAIL
  the                    sum = 0.916030534351  FAIL
  cat                    sum = 0.731707317073  FAIL
  sat                    sum = 0.784313725490  FAIL
  on                     sum = 0.784313725490  FAIL
  mat                    sum = 0.645161290323  FAIL
  rug                    sum = 0.645161290323  FAIL
  dog                    sum = 0.731707317073  FAIL
  ran                    sum = 0.645161290323  FAIL
  to                     sum = 0.645161290323  FAIL
  park                   sum = 0.645161290323  FAIL

Test passed? False


## Part VIII — Predicting the Next Word

$P(X_{t+1}\mid X_t = w)$ and $\arg\max_w$ for several contexts:

In [11]:
for ctx in [START, "the", "cat", "dog", "sat", "ran", "on", "to", "mat"]:
    dist = lm1.distribution(ctx)
    shown = ", ".join(f"{w}: {p:.3f}" for w, p in sorted(dist.items(), key=lambda kv: -kv[1]))
    best = lm1.argmax(ctx)
    tie = f"   (tie between {best})" if len(best) > 1 else ""
    print(f"{ctx:<8} -> argmax = {lm1.predict(ctx):<6} | {shown}{tie}")

<START>  -> argmax = the    | the: 1.000
the      -> argmax = cat    | cat: 0.250, dog: 0.250, mat: 0.167, rug: 0.167, park: 0.167   (tie between ['cat', 'dog'])
cat      -> argmax = sat    | sat: 0.667, ran: 0.333
dog      -> argmax = sat    | sat: 0.667, ran: 0.333
sat      -> argmax = on     | on: 1.000
ran      -> argmax = to     | to: 1.000
on       -> argmax = the    | the: 1.000
to       -> argmax = the    | the: 1.000
mat      -> argmax = <END>  | <END>: 1.000


### Question 9 — Are the most probable predictions what you would expect?

Partly. `sat → on`, `ran → to`, `on → the` and `mat → <END>` match intuition, but several predictions don't:

- **After `the` the argmax is a tie** between `cat` and `dog` (0.25 each), broken alphabetically. A person reading "the cat sat on **the** …" expects `mat` or `rug`, but the model sees only the single word `the`. It cannot tell subject position from object position, so it predicts `cat` there too and gives `park` (1/6) equal weight with `mat` and `rug`, even though "sat on the park" is odd.
- **After `sat` the model is certain** (`on`, probability 1). A person would also accept "sat down", "sat quietly", or the end of the sentence. The model treats everything it has not seen as impossible.
- **After `the` a person would never say "the <END>" or "the sat"**. The model agrees, but only because these did not occur, not because it knows any grammar.

The difference: a **probability model** is a summary of the *frequencies in its training data* under its structural assumption (here, one word of context). It has no semantics, world knowledge, or grammar beyond what the counts encode, and its "expectation" is relative to a six-sentence corpus. **Human expectations** use the whole sentence, meaning, and far more linguistic experience, and they rarely assign exactly 0 or 1. The model can be perfectly correct as an estimator of $P(X_t\mid X_{t-1})$ and still disagree with a person, because it answers a narrower question. The argmax is also only one point of a distribution: the model's real prediction is the whole row.

## Part IX — Generate Text

Ancestral sampling through the BN: $X_1 \sim P(X_1 \mid \texttt{<START>})$, $X_2 \sim P(X_2 \mid X_1)$, …, until `<END>` is sampled. We generate 20 sentences and save them to `outputs/`.

In [12]:
training_set = {" ".join(t) for t in tokenised}

def generate_many(model, n, mode="sample", seed=SEED, max_tokens=20):
    rng = random.Random(seed)
    return [model.generate(mode=mode, rng=rng, max_tokens=max_tokens) for _ in range(n)]

def describe(words, finished):
    s = " ".join(words)
    tags = []
    if not finished:
        tags.append("hit max_tokens")
    if finished and s not in training_set:
        tags.append("NEW")
    return s + (f"   [{', '.join(tags)}]" if tags else "")

samples_1 = generate_many(lm1, 20)
lines_1 = [f"{i:>2}. {describe(w, f)}" for i, (w, f) in enumerate(samples_1, 1)]
print("\n".join(lines_1))

(OUT_DIR / "generated_first_order.txt").write_text(
    "First-order model, sampling mode, 20 sentences (seed=%d)\n\n" % SEED + "\n".join(lines_1) + "\n")

 1. the cat ran to the dog ran to the rug   [NEW]
 2. the park   [NEW]
 3. the cat sat on the mat
 4. the dog ran to the mat   [NEW]
 5. the dog ran to the rug   [NEW]
 6. the dog sat on the dog sat on the mat   [NEW]
 7. the cat sat on the mat
 8. the mat   [NEW]
 9. the cat ran to the dog sat on the cat sat on the rug   [NEW]
10. the park   [NEW]
11. the cat sat on the cat sat on the dog sat on the mat   [NEW]
12. the rug   [NEW]
13. the rug   [NEW]
14. the rug   [NEW]
15. the dog ran to the dog sat on the dog ran to the dog sat on the rug   [NEW]
16. the rug   [NEW]
17. the park   [NEW]
18. the mat   [NEW]
19. the rug   [NEW]
20. the mat   [NEW]


716

`NEW` marks a sentence that does not appear in the training data. Only 2 of the 20 are training sentences, and the model's failures come in three kinds:

- **Two-word fragments** (`the rug`, `the park`, `the mat`) are the most common output. After the *first* `the`, the model picks an object noun half the time ($3 \times \tfrac16$), and every object noun is followed by `<END>`. It cannot know it is at the start of a sentence.
- **Wrong combinations**: "the dog ran to the mat" / "the dog ran to the rug". Every bigram is real, but "ran to the" should lead to `park`.
- **Recursion**: "the cat ran to the dog sat on the cat sat on the rug". Since `on → the → cat` is a legal path, the chain can re-enter the subject position indefinitely, and sentences can get arbitrarily long.

## Part X — Deterministic vs Probabilistic Generation

The same `generate` method, in both modes, five sentences each.

In [13]:
print("Mode A: greedy (argmax)")
for i, (w, f) in enumerate(generate_many(lm1, 5, mode="greedy"), 1):
    print(f"  {i}. {describe(w, f)}")

print("\nMode B: sampling")
for i, (w, f) in enumerate(generate_many(lm1, 5, mode="sample", seed=SEED + 1), 1):
    print(f"  {i}. {describe(w, f)}")

Mode A: greedy (argmax)
  1. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on   [hit max_tokens]
  2. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on   [hit max_tokens]
  3. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on   [hit max_tokens]
  4. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on   [hit max_tokens]
  5. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on   [hit max_tokens]

Mode B: sampling
  1. the rug   [NEW]
  2. the rug   [NEW]
  3. the cat ran to the dog ran to the park   [NEW]
  4. the cat sat on the dog sat on the dog sat on the cat sat on the mat   [NEW]
  5. the cat sat on the mat


In [14]:
# Why greedy never ends: follow the argmax chain from <START> and look for a cycle.
path, ctx = [START], START
while ctx not in path[:-1]:
    ctx = lm1.predict(ctx)
    path.append(ctx)
print(" -> ".join(path))
print(f"\nThe argmax chain revisits {ctx!r} without passing through <END>, so greedy decoding loops forever.")

<START> -> the -> cat -> sat -> on -> the

The argmax chain revisits 'the' without passing through <END>, so greedy decoding loops forever.


### Question 10 — Which mode produces more variation, and why?

**Sampling** produces all the variation. **Greedy decoding produces the same output five times**, and in this model that output is not even a sentence:

- Greedy is a deterministic function of the context. Starting from the same `<START>` with the same CPT it must follow the same path every time, so there is exactly one possible output.
- That path is `the → cat → sat → on → the → cat → …`, a **cycle**. `<END>` can only follow `mat`/`rug`/`park`, and none of those is ever the argmax after `the` (1/6 < 1/4). Greedy decoding therefore never terminates and has to be cut off by `max_tokens`. A first-order model has no memory that it has already used `the cat`.
- Sampling picks lower-probability branches in proportion to their probability (`dog` a quarter of the time after `the`, `ran` a third of the time after `cat`, `mat`/`rug`/`park` half the time after `the`). So it reaches `<END>` with probability 1 and produces a different sentence on almost every run. The five samples above are four different sentences, from two words to eighteen. The variety comes from the entropy of the CPT rows: deterministic rows (`sat → on`) contribute none, and `the` contributes most.

Greedy maximises each step locally, which is not the same as finding the most probable sentence. Under this model the most probable complete sentences are the fragments `the mat` / `the rug` / `the park` ($P = 1\cdot\tfrac16\cdot 1 = \tfrac16$ each, Part XIII). The best "real" sentence, "the cat sat on the mat", only gets $1\cdot\tfrac14\cdot\tfrac23\cdot1\cdot1\cdot\tfrac16\cdot1 = \tfrac1{36}$. Greedy produces neither, because choosing `cat` (1/4) over `mat` (1/6) at step 2 commits it to the loop.

## Part XI — A Second-Order Bayesian Network

Now each token has two parents, $X_{t-2} \rightarrow X_t \leftarrow X_{t-1}$, and

$$P(X_1,X_2,X_3,X_4) = P(X_1)\,P(X_2\mid X_1)\,P(X_3\mid X_1,X_2)\,P(X_4\mid X_2,X_3).$$

(With two `<START>` tokens of padding, $P(X_1) = P(X_1 \mid \texttt{<START>},\texttt{<START>})$ and $P(X_2\mid X_1) = P(X_2\mid \texttt{<START>}, X_1)$, so every factor has the same form.)

### Question 11 — How does the second-order model differ from the first-order model?

1. **Graph structure.** First-order is a chain: each node has one parent. Second-order adds a "skip" edge $X_{t-2}\to X_t$, so every node from $X_3$ on has two parents and the graph is no longer a simple chain. The independence assumption weakens to $X_t \perp \{X_1..X_{t-3}\} \mid X_{t-2}, X_{t-1}$.
2. **CPT.** The table is indexed by the *pair* of previous words. It has $|V|^2$ possible rows instead of $|V|$, each still a distribution over $|V|$ next words, and it is estimated from **triple counts**: $P(w\mid u,v) = C(u,v,w)/\sum_k C(u,v,w_k)$.
3. **Context.** Two words instead of one. The model can now tell `on the` (object position: `mat`/`rug`) from `<START> the` (subject position: `cat`/`dog`) and `to the` (`park`), which the first-order model conflated into a single row for `the`.
4. **Data needed.** Many more parameters ($|V|^2(|V|-1)$ vs $|V|(|V|-1)$ free parameters), and the same corpus is split across many more contexts, so each row rests on fewer observations. Much more data is needed for reliable estimates, and many more contexts are never observed at all (Part XIII quantifies this).

## Part XII — Use the LLM Again

**What should change in the probabilistic model, stated before accepting any code:**

- the conditioning set becomes $(X_{t-2}, X_{t-1})$: CPT rows are keyed by a *tuple* of two tokens;
- counts are of observed **triples** $C(u, v, w)$, and each row is normalised over $w$;
- sentences are padded with **two** `<START>` tokens so the first two words have a full context;
- generation keeps a history and uses its **last two** tokens as context;
- nothing else changes: prediction (argmax), sampling, stopping at `<END>`, and the normalisation test are the same, which also gives us a way to test it.

Prompt used:

> Modify the existing first-order autoregressive model into a second-order model. The model should estimate $P(X_t \mid X_{t-2}, X_{t-1})$. Represent the model using counts of observed triples and use these counts to construct conditional probability distributions. Pad each sentence with two `<START>` tokens. Keep the same public methods (`fit`, `distribution`, `show`, `argmax`, `predict`, `sample_next`, `generate`) so both models can be tested with the same code. Do not replace the model with a neural network or a pretrained language model.

The accepted code subclasses the first-order model and overrides only the parts listed above:

In [15]:
class SecondOrderLM(FirstOrderLM):
    # Second-order autoregressive LM: (X_{t-2}, X_{t-1}) -> X_t.
    # counts[(u, v)][w] = C(u, v, w), the count of the observed triple.

    order = 2

    def context(self, history):
        return tuple(history[-2:])

    def fit(self, sentences):
        for tokens in sentences:
            for u, v, w in zip(tokens, tokens[1:], tokens[2:]):
                self.counts[(u, v)][w] += 1
        for ctx, row in self.counts.items():
            total = sum(row.values())
            self.probs[ctx] = {w: c / total for w, c in row.items()}
        return self


def ctx_str(ctx):
    return " ".join(ctx) if isinstance(ctx, tuple) else ctx

training_data_2 = [add_markers(t, order=2) for t in tokenised]
lm2 = SecondOrderLM().fit(training_data_2)
print(" ".join(training_data_2[0]))

<START> <START> the cat sat on the mat <END>


**Inspecting and testing the second-order code.** The triple loop `zip(tokens, tokens[1:], tokens[2:])` yields `len(tokens) - 2` triples, one per predicted token including `<END>`. Contexts are tuples. The same normalisation test is run, plus hand-checked values:

In [16]:
assert check_normalisation(lm2.probs, fmt=ctx_str), "second-order CPT is not normalised"
assert lm2.probs[(START, START)] == {"the": 1.0}
assert lm2.probs[("on", "the")] == {"mat": 0.5, "rug": 0.5}
assert lm2.probs[("to", "the")] == {"park": 1.0}
assert lm2.probs[("the", "cat")] == {"sat": 2 / 3, "ran": 1 / 3}
# One triple per predicted token: sentence length + 1 (for <END>).
assert sum(sum(r.values()) for r in lm2.counts.values()) == sum(len(t) + 1 for t in tokenised)
print("\nSecond-order CPT normalised and matches hand-computed values.")

  <START> <START>        sum = 1.000000000000  OK
  <START> the            sum = 1.000000000000  OK
  the cat                sum = 1.000000000000  OK
  cat sat                sum = 1.000000000000  OK
  sat on                 sum = 1.000000000000  OK
  on the                 sum = 1.000000000000  OK
  the mat                sum = 1.000000000000  OK
  the rug                sum = 1.000000000000  OK
  the dog                sum = 1.000000000000  OK
  dog sat                sum = 1.000000000000  OK
  dog ran                sum = 1.000000000000  OK
  ran to                 sum = 1.000000000000  OK
  to the                 sum = 1.000000000000  OK
  the park               sum = 1.000000000000  OK
  cat ran                sum = 1.000000000000  OK

Second-order CPT normalised and matches hand-computed values.


In [17]:
print("Second-order CPT (observed contexts only):\n")
observed_2 = sorted(lm2.counts, key=lambda c: (c[0] != START, c))
print_cpt(lm2, observed_2, next_tokens, fmt=ctx_str)

Second-order CPT (observed contexts only):

                     cat    dog    mat     on   park    ran    rug    sat    the     to  <END>
<START> <START>        ·      ·      ·      ·      ·      ·      ·      ·      1      ·      ·
<START> the          1/2    1/2      ·      ·      ·      ·      ·      ·      ·      ·      ·
cat ran                ·      ·      ·      ·      ·      ·      ·      ·      ·      1      ·
cat sat                ·      ·      ·      1      ·      ·      ·      ·      ·      ·      ·
dog ran                ·      ·      ·      ·      ·      ·      ·      ·      ·      1      ·
dog sat                ·      ·      ·      1      ·      ·      ·      ·      ·      ·      ·
on the                 ·      ·    1/2      ·      ·      ·    1/2      ·      ·      ·      ·
ran to                 ·      ·      ·      ·      ·      ·      ·      ·      1      ·      ·
sat on                 ·      ·      ·      ·      ·      ·      ·      ·      1      ·      ·
the ca

In [18]:
print("Selected second-order contexts vs the first-order row for the same last word:\n")
for ctx in [(START, "the"), ("on", "the"), ("to", "the"), ("the", "cat")]:
    lm2.show(ctx)
    print()
lm1.show("the")

Selected second-order contexts vs the first-order row for the same last word:

P(. | ('<START>', 'the'))   [6 observations]
   cat       3/6  = 0.5000
   dog       3/6  = 0.5000

P(. | ('on', 'the'))   [4 observations]
   mat       2/4  = 0.5000
   rug       2/4  = 0.5000

P(. | ('to', 'the'))   [2 observations]
   park      2/2  = 1.0000

P(. | ('the', 'cat'))   [3 observations]
   sat       2/3  = 0.6667
   ran       1/3  = 0.3333

P(. | the)   [12 observations]
   cat       3/12 = 0.2500
   dog       3/12 = 0.2500
   mat       2/12 = 0.1667
   park      2/12 = 0.1667
   rug       2/12 = 0.1667


In [19]:
print("Second-order generation: greedy")
for i, (w, f) in enumerate(generate_many(lm2, 5, mode="greedy"), 1):
    print(f"  {i}. {describe(w, f)}")

samples_2 = generate_many(lm2, 20)
lines_2 = [f"{i:>2}. {describe(w, f)}" for i, (w, f) in enumerate(samples_2, 1)]
print("\nSecond-order generation: sampling (20)")
print("\n".join(lines_2))

(OUT_DIR / "generated_second_order.txt").write_text(
    "Second-order model, sampling mode, 20 sentences (seed=%d)\n\n" % SEED + "\n".join(lines_2) + "\n")

Second-order generation: greedy
  1. the cat sat on the mat
  2. the cat sat on the mat
  3. the cat sat on the mat
  4. the cat sat on the mat
  5. the cat sat on the mat

Second-order generation: sampling (20)
 1. the cat ran to the park
 2. the cat sat on the rug
 3. the cat sat on the mat
 4. the dog ran to the park
 5. the dog ran to the park
 6. the dog sat on the rug
 7. the cat sat on the mat
 8. the cat sat on the mat
 9. the cat ran to the park
10. the dog sat on the rug
11. the cat ran to the park
12. the dog sat on the mat
13. the cat sat on the rug
14. the cat sat on the rug
15. the dog ran to the park
16. the cat ran to the park
17. the dog ran to the park
18. the cat ran to the park
19. the cat sat on the mat
20. the cat sat on the mat


609

## Part XIII — Comparing the Two Models

Measured on the same corpus:

- **distinct parameters**: the full CPT size (every possible context × next token), the free parameters (each row has $|V|-1$ because it must sum to 1), and the number of *non-zero* entries actually estimated from data;
- **zero-probability contexts**: possible contexts that never occur in the training data, so their row is empty;
- **diversity**: over 1000 sampled sentences, the number of distinct sentences, how many are not in the training set, and the length distribution;
- **coherence**: examples, plus the probability each model assigns to a few test sentences.

In [20]:
def possible_contexts(order):
    # Every context the model could condition on: <START> padding followed by words.
    if order == 1:
        return [START] + vocab
    return [(START, START)] + [(START, w) for w in vocab] + [(u, v) for u in vocab for v in vocab]

def stats(model, n=1000):
    ctxs = possible_contexts(model.order)
    samples = generate_many(model, n, seed=SEED + 2, max_tokens=50)
    sentences = [" ".join(w) for w, f in samples if f]
    lengths = [len(w) for w, f in samples if f]
    return {
        "possible contexts": len(ctxs),
        "observed contexts": sum(1 for c in ctxs if c in model.probs),
        "zero-probability (unseen) contexts": sum(1 for c in ctxs if c not in model.probs),
        "full CPT entries": len(ctxs) * len(next_tokens),
        "free parameters": len(ctxs) * (len(next_tokens) - 1),
        "non-zero estimated entries": sum(len(r) for r in model.probs.values()),
        "deterministic rows (p = 1)": sum(1 for r in model.probs.values() if len(r) == 1),
        "distinct sentences / 1000": len(set(sentences)),
        "novel sentences (not in training)": len(set(sentences) - training_set),
        "mean length": round(sum(lengths) / len(lengths), 2),
        "max length": max(lengths),
    }

s1, s2 = stats(lm1), stats(lm2)
print(f"{'':<36}{'first-order':>12}{'second-order':>14}")
for k in s1:
    print(f"{k:<36}{s1[k]:>12}{s2[k]:>14}")

                                     first-order  second-order
possible contexts                             11           111
observed contexts                             11            15
zero-probability (unseen) contexts             0            96
full CPT entries                             121          1221
free parameters                              110          1110
non-zero estimated entries                    17            19
deterministic rows (p = 1)                     8            11
distinct sentences / 1000                    154             6
novel sentences (not in training)            148             0
mean length                                 5.78           6.0
max length                                    42             6


In [21]:
def sequence_probability(model, sentence):
    # Chain rule: P(x_1..x_T, <END>) = prod_t P(x_t | context_t).
    history = [START] * model.order
    p = 1.0
    for w in tokenise(sentence) + [END]:
        p *= model.distribution(model.context(history)).get(w, 0.0)
        history.append(w)
    return p

tests = [
    "the cat sat on the mat",                 # in training data
    "the dog ran to the park",                # in training data
    "the cat ran to the mat",                 # plausible, unseen
    "the cat sat on the park",                # odd, unseen
    "the cat sat on the dog sat on the rug",  # recursive, ungrammatical
    "the mat sat on the cat",                 # nonsense
]
print(f"{'sentence':<42}{'P first-order':>15}{'P second-order':>16}")
for s in tests:
    print(f"{s:<42}{sequence_probability(lm1, s):>15.5f}{sequence_probability(lm2, s):>16.5f}")

sentence                                    P first-order  P second-order
the cat sat on the mat                            0.02778         0.16667
the dog ran to the park                           0.01389         0.16667
the cat ran to the mat                            0.01389         0.00000
the cat sat on the park                           0.02778         0.00000
the cat sat on the dog sat on the rug             0.00463         0.00000
the mat sat on the cat                            0.00000         0.00000


In [22]:
# Novel first-order sentences, i.e. what the extra context removes.
rng = random.Random(SEED + 3)
novel = Counter()
for _ in range(2000):
    w, f = lm1.generate(rng=rng, max_tokens=50)
    s = " ".join(w)
    if f and s not in training_set:
        novel[s] += 1
print("Most frequent novel first-order sentences:")
for s, c in novel.most_common(8):
    print(f"  {c:>4}×  {s}")

Most frequent novel first-order sentences:
   350×  the park
   325×  the mat
   305×  the rug
    70×  the dog sat on the park
    48×  the cat sat on the park
    33×  the dog ran to the rug
    33×  the cat ran to the rug
    30×  the cat ran to the mat


**Summary of the comparison**

| | first-order | second-order |
|---|---|---|
| Full CPT size / free parameters | 121 / 110 | 1221 / 1110 |
| Non-zero entries actually estimated | 17 | 19 |
| Deterministic rows (a single next word, p = 1) | 8 of 11 | 11 of 15 |
| Unseen (zero-probability) contexts | 0 of 11 | 96 of 111 |
| Distinct sentences in 1000 samples | 154 (unbounded in principle, because of recursion) | 6 |
| Novel sentences (not in training data) | 148 | 0: it only reproduces training sentences |
| Mean / max sentence length | 5.78 / 42 words | 6.0 / 6 words |
| Greedy decoding | loops forever | "the cat sat on the mat" |

The *number of estimated* (non-zero) parameters barely changes, 17 → 19, even though the table is 10× larger. Six sentences simply do not contain more than about 20 distinct patterns. The extra capacity of the second-order model goes almost entirely into unseen contexts.

**Coherence, from the examples.** The first-order model's most frequent novel outputs are the fragments "the park" / "the mat" / "the rug" (about half of all samples), followed by "the dog sat on the park", "the cat ran to the rug", and long recursive chains up to 42 words. These are locally fluent (every bigram is real) but globally incoherent, because after `the` it has forgotten whether it is at the subject or the object. The probability table shows this directly: the first-order model gives "the cat sat on the **park**" exactly the same probability as "the cat sat on the **mat**" (1/36), and rates the fragment "the mat" six times higher than either. The second-order model only ever outputs the six training sentences, all perfectly coherent. With so little data, two words of context identify almost the whole sentence: 11 of its 15 observed contexts are deterministic. It is coherent because it has memorised the corpus, not because it has learned more English. It also assigns probability 0 to the perfectly reasonable "the cat ran to the mat", which the first-order model accepts.

### Question 12 — Why does more context help, and why does it make estimation harder?

**Why it helps.** Conditioning on more of the past can only reduce uncertainty on average, $H(X_t\mid X_{t-1},X_{t-2}) \le H(X_t\mid X_{t-1})$, and it brings the model closer to the true chain-rule factor $P(X_t\mid X_1,\dots,X_{t-1})$. Here the extra word separates the three different uses of `the` (`<START> the`, `on the`, `to the`). That removes the fragments, the wrong prepositions ("sat on the park") and the greedy loop, and it raises the probability of every training sentence from about 1/36 to 1/6.

**Why it hurts with limited data.** The CPT grows exponentially in the context length $k$: it has $|V|^k$ rows and $|V|^k(|V|-1)$ free parameters (110 → 1110 here, with only 10 words). The number of training tokens $N$ is fixed, so the average number of observations per context falls like $N/|V|^k$. As a result:
- most contexts are never observed (96 of 111 here), so the model has no prediction at all for them;
- observed contexts are seen once or twice, so their maximum-likelihood rows are extreme (mostly probability 1) and high-variance;
- unseen-but-valid continuations get probability 0 ("the cat ran to the mat"), so the model overfits and memorises the training set rather than generalising.

This is a bias–variance trade-off. Short context means a biased model (wrong independence assumption) with well-estimated parameters; long context means a less biased model with poorly estimated parameters. Real n-gram models deal with it using smoothing and back-off. Neural LMs deal with it by sharing parameters across contexts (Part XIV).

## Part XIV — The Connection to Modern Language Models

A modern LM uses the same exact factorisation with the first term read as $P(x_1\mid\texttt{<START>})$:

$$P(x_1,\dots,x_T) = \prod_{t=1}^{T} P(x_t\mid x_1,\dots,x_{t-1}).$$

Our `sequence_probability` function *is* this product. The only difference is how each factor is represented:

| | Simple BN model (this notebook) | Modern autoregressive LM |
|---|---|---|
| Representation | CPT (`self.probs`, a dict of dicts) | neural network $f_\theta(x_{<t})$ followed by softmax |
| Context | fixed window (1 or 2 tokens) | long learned context (thousands of tokens) |
| Parameters | one explicit probability per (context, token) | shared weights, far fewer than $|V|^k$ |
| Learning | counting + normalising (MLE) | gradient descent on the same log-likelihood |
| Generation | `sample_next` in a loop until `<END>` | the same loop (plus temperature / top-p) |

Because the network shares parameters across contexts, a context it has never seen still gets a sensible distribution from similar ones. That addresses the sparsity problem of Question 12 without an exponentially large table.

## Part XV — Reflection on the Role of the LLM

### Question 13 — Why is Approach B preferable to Approach A?

"Write a Python language model for me" (A) hands the LLM the modelling decisions. "Implement $P(X_t\mid X_{t-1})$ from transition counts, with sampling-based generation" (B) keeps them with me and asks the LLM only for code.

- **Specifying the intended behaviour.** B pins down the variables, the dependency structure, the estimator, and the generation procedure. With A the LLM could return anything from a bigram model to a call to a pretrained transformer, and neither of us could say whether it is "correct", because correct was never defined.
- **Understanding the representation.** Because I know the model is a CPT of normalised bigram counts, I know where to look (`self.counts`, `self.probs`) and what the numbers should be ($P(\texttt{cat}\mid\texttt{the}) = 3/12$). That is what made Questions 4–7 answerable.
- **Validating the generated implementation.** A specification can be checked. I compared the code's counts against hand-computed values and found behaviour the spec did not cover (non-termination, silent tie-breaking, `KeyError` on unseen words), which I then added to the spec.
- **Testing probabilistic invariants.** B implies properties that must hold whatever the code looks like: every row sums to 1, the probabilities are in [0,1], the counts add up to the number of transitions, and the joint probability equals the product of the factors. These catch real bugs (Q8) that inspecting outputs would not, because `random.choices` hides a non-normalised table.
- **Distinguishing implementation from model.** The *model* is the BN and its CPT; the *implementation* is one of many programs that realise it. B keeps them separate: the second-order change was stated as a change to the model (condition on two parents, count triples) before any code was accepted, and the same tests validated both implementations. A merges the two, so a bug in the code cannot be told apart from a flaw in the model.

### Question 14 — What did thinking of the language model as a Bayesian network give you?

- **A representation of dependencies.** The graph $X_{t-1}\to X_t$ makes explicit what each word is allowed to depend on. Adding one edge ($X_{t-2}\to X_t$) *is* the step from first- to second-order, and the rest (the CPT shape, the counts needed) follows from the graph.
- **A factorisation of the joint distribution.** The BN turns $P(X_1,\dots,X_T)$ into a product of local CPTs. That is what let me compute sentence probabilities (`sequence_probability`) and explain why one unseen bigram makes a whole sentence's probability 0.
- **A way to interpret conditional probabilities.** Each CPT entry has a concrete meaning, "the fraction of times `the` was followed by `cat`", so the model's predictions can be read and questioned directly (Q9: why `park` gets 1/6 after "sat on the").
- **A principled method for generation.** Ancestral sampling in topological order ($X_1$, then $X_2$ given its parent, …) is the standard way to sample a BN, and it draws exactly from the joint. It also explained why greedy decoding is *not* sampling from the model and why it can loop.
- **A way to reason about independence assumptions and the effect of context.** The Markov assumption $X_t\perp X_{<t-1}\mid X_{t-1}$ explained the incoherent first-order sentences, and counting CPT parameters ($|V|^k(|V|-1)$) explained why the second-order model memorised the data.
- **A way to test whether the implementation matches its specification.** Each CPT row must be a distribution, so normalisation is a property test that follows directly from the BN definition. It validated both LLM-assisted implementations and would catch the 0.87 bug of Q8.

### Reflection: how the LLM was used and how its output was validated

I used an LLM (Claude) as a code generator, not as the designer. Before prompting I wrote down the variables, the graph, and the estimator (Parts I–IV, including the hand counts), and each prompt was a behavioural spec (Parts V and XII). I did not accept any generated code until it passed the tests in Part VII and matched the hand-computed CPT.

**Example of LLM-generated code I inspected and corrected.** The first version of the generator followed my first prompt ("stop when `<END>` is generated") literally:

```python
def generate(self, mode="sample", rng=None):
    rng = rng or random.Random()
    prev, words = START, []
    while True:
        nxt = self.predict(prev) if mode == "greedy" else self.sample_next(prev, rng)
        if nxt == END:
            break
        words.append(nxt)
        prev = nxt
    return words
```

On inspection there were three problems:

1. **Greedy mode never terminates.** The argmax chain is the cycle `the → cat → sat → on → the` (demonstrated in Part X), so `nxt == END` is never true and the `while True` loop hangs. Sampling mode is fine, which is why a quick test in the default mode would not reveal it.
2. **`predict` hid ties.** It used `max(dist, key=dist.get)`, which silently returns whichever of `cat`/`dog` was inserted first. The result was right by accident but depended on dictionary order. I made ties explicit (`argmax` returns all of them) and the tie-break deterministic and documented.
3. **Unseen contexts crashed.** `self.probs[prev]` raised `KeyError` for a word not in the training data, and if `sample_next` returned `None` the loop would have appended `None` to the sentence.

The fix was to **change the spec, not just patch the code**: I added the `max_tokens` bound with a `finished` flag, explicit tie reporting, and an empty distribution for unseen contexts (items 5, 7 and 8 of the Part V prompt). I then kept the tests that exposed the problems. The second-order model was checked the same way: triple counts, the normalisation test, hand-checked rows like $P(\cdot\mid\texttt{on the}) = \{\texttt{mat}: \tfrac12, \texttt{rug}: \tfrac12\}$, and the observation that its greedy output terminates while the first-order one does not.

---

### Deliverables checklist

| # | Deliverable | Where |
|---|---|---|
| 1 | First-order implementation | `FirstOrderLM` (Part V) |
| 2 | Second-order implementation | `SecondOrderLM` (Part XII) |
| 3 | CPTs for selected contexts | Part IV (Q3) and Part XII |
| 4 | Generated text | Parts IX, X, XII; `outputs/generated_first_order.txt`, `outputs/generated_second_order.txt` |
| 5 | Normalisation test results | Part VII and Part XII |
| 6 | Answers to Questions 1–14 | inline, next to each part; also collected in [`../answers-to-questions/answers.md`](../answers-to-questions/answers.md) |
| 7 | Reflection on LLM use | this section |